# DAgger as a loss in DDPG — Decaying weight, adaptively normalized, and filtered vs the baseline

Implements three interpretations from `dagger_as_loss_in_ddpg.md` (§4) on top of
standard DDPG, and compares **training env steps to combined fulfillment > 0.7**
against pure DDPG. In all three methods the **student always drives** (Gaussian
exploration noise around π); the MPPI expert only *labels* visited states, and
each method controls the BC term differently:

| Method | §4 | Actor loss | Expert queries |
|---|---|---|---|
| `1. decay` | **additive, decaying weight** | `(1 − Q) + λ_bc(t)·BC`, `λ_bc(t) = 1.0 · 0.5^(t/5000)` | every step while λ_bc ≥ 0.01 (≈ first 33k steps) |
| `2. adaptive` | **additive, adaptively normalized** (TD3+BC) | `−λ·Q + BC`, `λ = α / mean(‖Q(s,a)‖)`, α = 2.5 — the Q term is rescaled so the Q:BC ratio means the same thing at every point in training | every 5th step, whole run |
| `3. filtered` | **filtered (Q-filter)**: the expert only gets a vote when it is right | `(1 − Q) + BC·𝟙[Q(s, a*) > Q(s, π(s))]` — self-annealing, state-conditional | every 5th step, whole run |
| `baseline` | — | `(1 − Q)` — pure DDPG | never |

Notes on fidelity to the document:
- (1) is exponential in env steps (1 update/step, so update counter ≡ step
  counter), with the same query cutoff as before (λ_bc < 0.01 ⇒ BC ≈ 0 ⇒ stop
  paying for plans).
- (2) normalizes the **Q term** (as TD3+BC does) using the batch mean |Q(s,a)|
  on behaviour actions, α = 2.5.
- (3) is the **hard** Nair et al. Q-filter; the BC sum is normalized by the
  number of *labeled* samples in the batch, so as the filter closes the term
  genuinely shrinks (the self-annealing the document describes). Per §4.V the
  filter is noisy while the critic is warm — that cost is accepted as-is.
- For (2) (3) the expert votes for the whole run, so the query cost is
  controlled by **thinning** (label every 5th visited state — §5.4's
  query-budget lens) rather than by a cutoff. Labels live in the replay buffer
  with a mask; BC only ever samples labeled transitions' loss.

**Procedure**: `hopper`, `walker`, `g1_standup`; budget **500k env steps**;
stop when the combined mean of all fulfillment terms (horizon-normalized,
5 greedy episodes) exceeds **0.7**; eval every 5k steps; seed 0; report
steps-to-0.7 and final per-term fulfillments.

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

DEVICE = "cpu"  # CPU torch build; these network sizes train fine here


# ======================================================================== #
# Networks: deterministic actor + scalar Q critic in [0,1]
# ======================================================================== #
def mlp(sizes, act=nn.ReLU, out_act=nn.Identity):
    layers = []
    for i in range(len(sizes) - 1):
        layers += [nn.Linear(sizes[i], sizes[i + 1]),
                   act() if i < len(sizes) - 2 else out_act()]
    return nn.Sequential(*layers)


class Actor(nn.Module):
    """Deterministic policy  pi(s) -> action in [-1, 1]  (tanh output)."""
    def __init__(self, obs_dim, act_dim, hidden=(256, 256)):
        super().__init__()
        self.net = mlp([obs_dim, *hidden, act_dim], out_act=nn.Tanh)

    def forward(self, s):
        return self.net(s)


class QCritic(nn.Module):
    """Scalar  Q(s,a) in [0,1]  (sigmoid head). The normalized TD target
    y = (1-g)*r + g*Q' stays in [0,1] when r does, so (1 - Q) in the actor
    loss is on the same scale as the BC term."""
    def __init__(self, obs_dim, act_dim, hidden=(256, 256)):
        super().__init__()
        self.net = mlp([obs_dim + act_dim, *hidden, 1])

    def forward(self, s, a):
        return torch.sigmoid(self.net(torch.cat([s, a], dim=-1)))


print("ready | torch", torch.__version__)

ready | torch 2.13.0+cu130


In [2]:
# ======================================================================== #
# Replay buffer — DDPG transitions + optional MPPI label (with a mask)
# ======================================================================== #
class ReplayBuffer:
    def __init__(self, cap, obs_dim, act_dim):
        self.cap = cap
        self.s     = np.zeros((cap, obs_dim), np.float32)
        self.a     = np.zeros((cap, act_dim), np.float32)
        self.r     = np.zeros((cap, 1), np.float32)     # scalar fulfillment reward
        self.s2    = np.zeros((cap, obs_dim), np.float32)
        self.done  = np.zeros((cap, 1), np.float32)     # natural termination -> no bootstrap
        self.a_exp = np.zeros((cap, act_dim), np.float32)  # MPPI label (if any)
        self.m     = np.zeros((cap, 1), np.float32)     # 1 if this row carries a label
        self.idx, self.full = 0, False

    def add(self, s, a, r, s2, done, a_exp=None):
        j = self.idx
        self.s[j], self.a[j], self.r[j] = s, a, r
        self.s2[j], self.done[j] = s2, done
        if a_exp is not None:
            self.a_exp[j], self.m[j] = a_exp, 1.0
        else:
            self.a_exp[j], self.m[j] = 0.0, 0.0
        self.idx = (self.idx + 1) % self.cap
        self.full = self.full or self.idx == 0

    def sample(self, n):
        hi = self.cap if self.full else self.idx
        k = np.random.randint(0, hi, size=n)
        t = lambda x: torch.as_tensor(x[k], device=DEVICE)
        return (t(self.s), t(self.a), t(self.r), t(self.s2), t(self.done),
                t(self.a_exp), t(self.m))

    def __len__(self):
        return self.cap if self.full else self.idx

In [ ]:
class DDPG:
    def __init__(self, obs_dim, act_dim, *, gamma=0.9, tau=0.005,
                 actor_lr=1e-3, critic_lr=1e-3, hidden=(256, 256)):
        self.gamma, self.tau = gamma, tau
        self.obs_dim, self.act_dim, self.hidden = obs_dim, act_dim, hidden
        self.actor    = Actor(obs_dim, act_dim, hidden).to(DEVICE)
        self.actor_t  = Actor(obs_dim, act_dim, hidden).to(DEVICE)
        self.critic   = QCritic(obs_dim, act_dim, hidden).to(DEVICE)
        self.critic_t = QCritic(obs_dim, act_dim, hidden).to(DEVICE)
        self.actor_t.load_state_dict(self.actor.state_dict())
        self.critic_t.load_state_dict(self.critic.state_dict())
        self.a_opt = torch.optim.Adam(self.actor.parameters(),  lr=actor_lr)
        self.c_opt = torch.optim.Adam(self.critic.parameters(), lr=critic_lr)

    @torch.no_grad()
    def act(self, s, sigma=0.0):
        """Greedy action, plus Gaussian action-space exploration noise if sigma>0."""
        s = torch.as_tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        a = self.actor(s).squeeze(0).cpu().numpy()
        if sigma > 0:
            a = np.clip(a + sigma * np.random.randn(self.act_dim), -1.0, 1.0)
        return a

    def train_step(self, buf, batch_size, method="baseline",
                   lambda_bc=0.0, alpha=2.5):
        """One DDPG update. `method` selects the BC methodology, with the following actor loss formulas:
          baseline: pi_loss = (1 - Q)                                (no BC)
          decay   : pi_loss = (1 - Q) + lambda_bc * BC               
          adaptive: pi_loss = -lam*Q + BC, lam = alpha/mean|Q(s,a)|  
          filtered: pi_loss = (1 - Q) + BC * 1[Q(s,a*) > Q(s,pi)]   
        BC is masked to labeled transitions and normalized by their count, so
        for `filtered` the term genuinely shrinks as the filter closes."""
        s, a, r, s2, done, a_exp, m = buf.sample(batch_size)

        # --- critic: normalized TD target keeps Q in [0,1] ---
        with torch.no_grad():
            y = (1 - self.gamma) * r + self.gamma * (1 - done) * \
                self.critic_t(s2, self.actor_t(s2))
        q = self.critic(s, a)
        c_loss = ((y - q) ** 2).mean()
        self.c_opt.zero_grad(); c_loss.backward(); self.c_opt.step()

        # --- actor ---
        pi = self.actor(s)
        q_pi = self.critic(s, pi)
        n_lab = m.sum().clamp_min(1.0)
        bc = ((pi - a_exp) ** 2).mean(dim=-1, keepdim=True) #action taken - expert action   # per-sample BC

        if method == "adaptive":                              
            lam = (alpha / q.abs().mean().clamp_min(1e-3)).detach()
            pi_loss = -(lam * q_pi).mean() + (bc * m).sum() / n_lab
        elif method == "filtered":                          
            with torch.no_grad():
                vote = (self.critic(s, a_exp) > q_pi).float() * m
            pi_loss = (1.0 - q_pi).mean() + (bc * vote).sum() / n_lab
        else:                                                 
            pi_loss = (1.0 - q_pi).mean()
            if lambda_bc > 0.0:
                pi_loss = pi_loss + lambda_bc * (bc * m).sum() / n_lab

        self.a_opt.zero_grad(); pi_loss.backward(); self.a_opt.step()

        # --- Polyak target update ---
        with torch.no_grad():
            for tp, sp in zip(self.actor_t.parameters(), self.actor.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * sp)
            for tp, sp in zip(self.critic_t.parameters(), self.critic.parameters()):
                tp.mul_(1 - self.tau).add_(self.tau * sp)
        return float(c_loss.detach()), float(pi_loss.detach())

In [ ]:
# === Experiment config + the shared environment wrapper =========================
import time
import mujoco
from analytic_mppi.tasks import make_task
from analytic_mppi.tasks.base import power_mean as np_power_mean
from analytic_mppi.dynamics import MujocoBackend
from analytic_mppi.controllers.mppi_v2 import MPPIv2

SEED          = 0
FPL_P         = 0.1        # repo MPPI default fpl_p; also collapses r_vec to the scalar reward
GAMMA         = 0.9
TARGET_F      = 0.7        # success threshold: COMBINED MEAN of the fulfillment terms
TOTAL_BUDGET  = 500_000    # env steps per run
EVAL_EVERY    = 5_000      # eval cadence (1k at this scale costs hours of pure eval)
BUFFER_CAP    = 500_000    # replay capacity for the 5M-step scale
HORIZON       = 400        # episode truncation
RESET_NOISE   = 5e-3       # gym-style uniform reset noise on qpos/qvel


ENV_CFG = {
    "hopper": dict(
        drop_qpos=(0,), healthy_z=0.5, init="zeros",
        teacher=dict(num_samples=256, noise_level=0.2, plan_horizon=1.0,
                     num_knots=15, spline_type="cubic", iterations=2)),
    "walker": dict(
        drop_qpos=(1,), healthy_z=0.8, init="zeros",
        teacher=dict(num_samples=256, noise_level=0.3, plan_horizon=0.6,
                     num_knots=8, spline_type="cubic", iterations=2)),
    "g1_standup": dict(
        drop_qpos=(0, 1), healthy_z=0.5, init="keyframe:stand",
        teacher=dict(num_samples=256, noise_level=0.2, plan_horizon=0.6,
                     num_knots=8, spline_type="cubic", iterations=2)),
}


class FPLEnv:
    """Gym-style wrapper: repo task + MujocoBackend, vector fulfillment reward.

    obs = [qpos minus the root-translation coords, qvel]
    reward = task.running_cost_terms_f in [0,1]^n_obj
    One backend serves closed-loop stepping (backend.data) AND the MPPI teacher's
    batched planning rollouts (separate _thread_data) without interference.
    """

    def __init__(self, name, seed=SEED):
        cfg = ENV_CFG[name]
        self.name = name
        self.task = make_task(name)
        self.backend = MujocoBackend(self.task.model_path)
        self.keep_qpos = np.array(
            [i for i in range(self.backend.nq) if i not in cfg["drop_qpos"]])
        self.obs_dim = len(self.keep_qpos) + self.backend.nv
        self.act_dim = self.backend.nu
        self.healthy_z = cfg["healthy_z"]
        self.rng = np.random.default_rng(seed)
        self.t = 0
        # base reset state: zeros (== default pose) or a named keyframe
        base = np.zeros(self.backend.nstate)
        if cfg["init"].startswith("keyframe:"):
            kf = self.backend.model.keyframe(cfg["init"].split(":", 1)[1])
            base[self.backend.qpos_slice] = kf.qpos
        self._base_state = base
        self.reset()
        self.n_obj = int(self._r_probe().shape[-1])

    def _obs(self):
        d = self.backend.data
        return np.concatenate([np.asarray(d.qpos)[self.keep_qpos],
                               d.qvel]).astype(np.float32)

    def _r_probe(self):
        d = self.backend.data
        return self.task.running_cost_terms_f(
            d.qpos, d.qvel, d.sensordata, np.zeros(self.act_dim))

    def torso_z(self):
        return float(self.task._torso_height(self.backend.data.sensordata))

    def reset(self):
        state = self._base_state.copy()
        state[self.backend.qpos_slice] += self.rng.uniform(
            -RESET_NOISE, RESET_NOISE, self.backend.nq)
        state[self.backend.qvel_slice] += self.rng.uniform(
            -RESET_NOISE, RESET_NOISE, self.backend.nv)
        self.backend.set_state(state)               # set_state runs mj_forward
        self.t = 0
        return self._obs()

    def step(self, u):
        u = np.clip(np.asarray(u, dtype=np.float64),
                    self.task.u_min, self.task.u_max)
        self.backend.step(u)
        # mj_step leaves data.sensordata at the PRE-step state; refresh before reading
        mujoco.mj_forward(self.backend.model, self.backend.data)
        d = self.backend.data
        r_vec = self.task.running_cost_terms_f(d.qpos, d.qvel, d.sensordata, u)
        self.t += 1
        terminated = self.torso_z() < self.healthy_z
        truncated = (not terminated) and (self.t >= HORIZON)
        return self._obs(), r_vec.astype(np.float32), terminated, truncated

    def full_state(self):
        """FULLPHYSICS state of the wrapper's exact current state (for the teacher)."""
        return self.backend.get_state()


for _n in ENV_CFG:
    _e = FPLEnv(_n)
    _e.reset()
    print(f"{_n:11s} obs_dim={_e.obs_dim:3d} act_dim={_e.act_dim:3d} "
          f"n_obj={_e.n_obj} dt={_e.backend.dt}  reset torso z={_e.torso_z():.3f}")

hopper      obs_dim= 11 act_dim=  3 n_obj=4 dt=0.02  reset torso z=1.245
walker      obs_dim= 17 act_dim=  6 n_obj=4 dt=0.01  reset torso z=1.298
g1_standup  obs_dim= 69 act_dim= 29 n_obj=3 dt=0.02  reset torso z=0.986


In [ ]:
# === The fulfillment metric: horizon-normalized per-term means ==================
def episode_terms_hnorm(R, horizon=None):
    """(T, n_obj) per-step fulfillments -> per-term  sum_t r_t,i / HORIZON.

    Normalising by the FULL horizon (not the survived length T) makes a fallen
    (terminated) episode score 0 for every missing step. Without this, g1_standup
    is degenerate: surviving-step averages are ~0.9 even for policies that fall
    after 2 s (the G1 is passively stable at zero torque)."""
    horizon = HORIZON if horizon is None else horizon
    return np.asarray(R, np.float32).sum(axis=0) / horizon


def evaluate_fulfillment(policy_fn, env_name, episodes=5, eval_seed=10_000):
    """Greedy rollouts on fresh, deterministically-seeded envs.
    Returns (combined mean across terms, per-term means (n_obj,))."""
    per = []
    for e in range(episodes):
        env = FPLEnv(env_name, seed=eval_seed + e)
        s, R = env.reset(), []
        done = False
        while not done:
            s, r_vec, term, trunc = env.step(policy_fn(s))
            R.append(r_vec)
            done = term or trunc
    # per-episode per-term horizon-normalized fulfillment, then mean over episodes
        per.append(episode_terms_hnorm(R))
    per_term = np.mean(per, axis=0)
    return float(per_term.mean()), per_term


# --- smoke test: one random-action episode on the hopper ------------------------
_env = FPLEnv("hopper", seed=SEED)
_rng_smoke = np.random.default_rng(1)
s = _env.reset()
assert s.shape == (11,)
T_smoke, done = 0, False
while not done:
    s, r_vec, term, trunc = _env.step(_rng_smoke.uniform(-1, 1, 3))
    assert r_vec.shape == (4,) and (r_vec >= 0).all() and (r_vec <= 1).all()
    T_smoke += 1
    done = term or trunc
print(f"smoke test: random-action episode ended after {T_smoke} steps "
      f"({'fell' if term else 'truncated'}), final torso z = {_env.torso_z():.3f}")

smoke test: random-action episode ended after 25 steps (fell), final torso z = 0.490


In [ ]:
# === Teacher: MPPIv2 with FPL cost terms ========================================
def make_teacher(env, seed=SEED):
    """Per-env MPPI config from ENV_CFG.

    fpl_gamma=0.99, NOT the standard GAMMA=0.9: the discount multiplies
    per-step fulfillment inside the plan horizon (0.9**60 ~ 0.001), so at 0.9
    the planner's effective lookahead is ~10 sim steps -- it tracks velocity
    but cannot see slow height collapse, and falls at t~80 (hopper closed-loop
    fulfillment 0.47). At 0.99 the documented teacher scores reproduce
    (hopper ~0.85, g1_standup ~0.89; verified 2026-07-15)."""
    return MPPIv2(env.task, env.backend, temperature=0.05, seed=seed,
                  use_fpl_cost=True, fpl_p=FPL_P, fpl_gamma=0.99,
                  **ENV_CFG[env.name]["teacher"])

## The training loop

In [7]:
# === Driver: one loop, four methods =============================================
METHODS     = ("baseline", "decay", "adaptive", "filtered")
LAMBDA0     = 1.0        # III: initial BC weight
HALF_LIFE   = 5_000      # III: env steps for lambda_bc to halve
LBC_CUTOFF  = 0.01       # III: below this, BC off and no more MPPI queries
LABEL_EVERY = 5          # IV/V: label every k-th visited state (query thinning)
ALPHA       = 2.5        # IV: TD3+BC normalization constant


def lambda_bc(t, lambda0=LAMBDA0, half_life=HALF_LIFE):
    return lambda0 * 0.5 ** (t / half_life)


def train_method_ddpg(env_name, method, *, budget=TOTAL_BUDGET, seed=SEED,
                      sigma=0.1, batch_size=256, eval_every=EVAL_EVERY,
                      target=TARGET_F, log=print):
    """Student-driven DDPG with the chosen BC interpretation (see cell above).
    Returns (hist, steps, labeled, final_per_term)."""
    assert method in METHODS
    np.random.seed(seed); torch.manual_seed(seed)
    env = FPLEnv(env_name, seed=seed + 500)
    ctrl = make_teacher(env, seed=seed) if method != "baseline" else None
    agent = DDPG(env.obs_dim, env.act_dim)
    buf = ReplayBuffer(BUFFER_CAP, env.obs_dim, env.act_dim)
    warmup = 1_000 if method == "baseline" else 0
    hist = {"eval_step": [], "eval_f": [], "eval_terms": []}
    labeled, steps, next_eval, stop = 0, 0, eval_every, False

    def do_eval(t):
        nonlocal stop
        f, per = evaluate_fulfillment(lambda s_: agent.act(s_), env_name)
        hist["eval_step"].append(t); hist["eval_f"].append(f)
        hist["eval_terms"].append(per)
        log(f"  [{env_name}/{method}] steps {t:7,d}  combined {f:.3f}  "
            f"terms {np.round(per, 3)}  labels {labeled:,}")
        if f > target:
            stop = True
            log(f"  [{env_name}/{method}] TARGET: combined {f:.3f} > {target} "
                f"at {t:,} env steps")

    def wants_label(t):
        if method == "decay":
            return lambda_bc(t) >= LBC_CUTOFF        # every step, until cutoff
        if method in ("adaptive", "filtered"):
            return t % LABEL_EVERY == 0              # thinned, whole run
        return False

    do_eval(0)
    while not stop and steps < budget:
        s = env.reset()
        if ctrl is not None:
            ctrl.reset()                             # zero the warm-start mean
        done_ep = False
        while not done_ep:
            a_exp = None
            if ctrl is not None and wants_label(steps):
                a_exp = np.clip(ctrl.act(env.full_state()),
                                env.task.u_min, env.task.u_max).astype(np.float32)
                labeled += 1
            a = (np.random.uniform(-1, 1, env.act_dim) if steps < warmup
                 else agent.act(s, sigma))
            s2, r_vec, term, trunc = env.step(a)
            buf.add(s, np.asarray(a, np.float32),
                    float(np_power_mean(r_vec, FPL_P, eps=1e-6)), s2,
                    float(term), a_exp)
            s = s2; steps += 1
            if len(buf) > batch_size and steps >= warmup:
                agent.train_step(buf, batch_size, method=method,
                                 lambda_bc=(lambda_bc(steps)
                                            if method == "decay" and
                                            lambda_bc(steps) >= LBC_CUTOFF
                                            else 0.0))
            if steps >= next_eval:
                next_eval += eval_every
                do_eval(steps)
            done_ep = term or trunc or stop or steps >= budget
    return hist, steps, labeled, hist["eval_terms"][-1]


def steps_to_target(hist, target=TARGET_F):
    """First step count at which the combined eval mean > target, else None."""
    for st, f in zip(hist["eval_step"], hist["eval_f"]):
        if f > target:
            return st
    return None


def run_method(env_name, method, log=print):
    t0 = time.perf_counter()
    hist, steps, labeled, per = train_method_ddpg(env_name, method, log=log)
    wall = time.perf_counter() - t0
    log(f"[{env_name}/{method}] {steps:,} env steps, {labeled:,} labels, "
        f"{wall:.0f}s")
    EXPERIMENTS.setdefault(env_name, {})[method] = dict(
        hist=hist, steps=steps, labeled=labeled, per_term=per, wall=wall)


EXPERIMENTS = {}   # env_name -> method -> result dict

In [ ]:
# ---- hopper: all four methods ---------------------------------------------------
for _m in METHODS:
    run_method("hopper", _m)
    print()

  [hopper/baseline] steps       0  combined 0.038  terms [0.009 0.053 0.032 0.058]  labels 0
  [hopper/baseline] steps   5,000  combined 0.306  terms [0.152 0.403 0.358 0.311]  labels 0
  [hopper/baseline] steps  10,000  combined 0.451  terms [0.234 0.639 0.451 0.48 ]  labels 0
  [hopper/baseline] steps  15,000  combined 0.634  terms [0.33  0.879 0.705 0.621]  labels 0
  [hopper/baseline] steps  20,000  combined 0.314  terms [0.201 0.404 0.322 0.328]  labels 0
  [hopper/baseline] steps  25,000  combined 0.751  terms [0.712 0.78  0.748 0.764]  labels 0
  [hopper/baseline] TARGET: combined 0.751 > 0.7 at 25,000 env steps
[hopper/baseline] 25,000 env steps, 0 labels, 891s

  [hopper/decay] steps       0  combined 0.038  terms [0.009 0.053 0.032 0.058]  labels 0


In [ ]:
# ---- walker: all four methods ----------------------------------------------------
for _m in METHODS:
    run_method("walker", _m)
    print()

In [ ]:
# ---- g1_standup (humanoid standup): all four methods -----------------------------
# The labeled methods pay ~45 ms of MPPI planning per label here.
for _m in METHODS:
    run_method("g1_standup", _m)
    print()

In [ ]:
# === Results: steps-to-0.7 across methods =======================================
LABELS = {"baseline": "pure DDPG", "decay": "III decay",
          "adaptive": "IV adaptive", "filtered": "V filtered"}
COLORS = {"baseline": "tab:red", "decay": "tab:blue",
          "adaptive": "tab:orange", "filtered": "tab:green"}
ENVS_RUN = [n for n in ("hopper", "walker", "g1_standup") if n in EXPERIMENTS]

print(f"{'env':12s}{'method':14s}{'steps to 0.7':>14s}{'labels':>10s}"
      f"{'best combined':>15s}{'final combined':>16s}")
for env_name in ENVS_RUN:
    for m in METHODS:
        if m not in EXPERIMENTS[env_name]:
            continue
        r = EXPERIMENTS[env_name][m]
        st = steps_to_target(r["hist"])
        st_str = f"{st:,}" if st is not None else "not reached"
        print(f"{env_name:12s}{LABELS[m]:14s}{st_str:>14s}{r['labeled']:>10,}"
              f"{max(r['hist']['eval_f']):>15.3f}"
              f"{float(np.mean(r['per_term'])):>16.3f}")

print("\nfinal per-term fulfillments:")
for env_name in ENVS_RUN:
    names = FPLEnv(env_name).task.cost_term_names_f
    for m in METHODS:
        if m not in EXPERIMENTS[env_name]:
            continue
        per = EXPERIMENTS[env_name][m]["per_term"]
        pairs = "  ".join(f"{n}={v:.3f}" for n, v in zip(names, per))
        print(f"  {env_name:12s}{LABELS[m]:14s}{pairs}")

fig, axes = plt.subplots(1, len(ENVS_RUN), figsize=(5.6 * len(ENVS_RUN), 3.8),
                         squeeze=False)
for col, env_name in enumerate(ENVS_RUN):
    ax = axes[0, col]
    for m in METHODS:
        if m not in EXPERIMENTS[env_name]:
            continue
        r = EXPERIMENTS[env_name][m]
        ax.plot(r["hist"]["eval_step"], r["hist"]["eval_f"], "-o", ms=2.5,
                color=COLORS[m], label=LABELS[m])
    ax.axhline(TARGET_F, color="k", ls="--", lw=1, alpha=0.6)
    ax.set_title(env_name)
    ax.set_xlabel("env steps")
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    if col == 0:
        ax.set_ylabel("combined mean fulfillment")
        ax.legend(loc="upper left", fontsize=8)

fig.suptitle("DAgger-as-loss interpretations vs pure DDPG "
             f"(seed={SEED}, budget={TOTAL_BUDGET:,}, stop at >{TARGET_F})",
             y=1.02)
fig.tight_layout()
plt.show()